# 🎓 Fine-tuning LoRA — Minci (Virtual Asisten Akademik STT Cipasung)

Notebook ini melatih **Llama 3.2 3B Instruct** dengan **LoRA** memakai **Unsloth** supaya gaya bahasanya jadi lebih luwes ala gen-z tapi tetap sopan.

**Cara pakai:**
1. Runtime → Change runtime type → pilih **T4 GPU**.
2. Jalankan semua cell dari atas ke bawah (Runtime → Run all), atau satu-satu.
3. Saat diminta upload, upload file `dataset_contoh.json` (atau dataset kamu sendiri dengan format yang sama).
4. Di akhir notebook, file `.gguf` hasil training akan otomatis ke-download ke laptop kamu.

Kamu **tidak perlu edit kode apapun** di notebook ini kecuali kalau mau ganti nama file dataset.

## 1. Install library yang dibutuhkan

In [ ]:
%%capture
import torch
!pip install unsloth
!pip install --upgrade --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git
!pip install --no-deps trl peft accelerate bitsandbytes

## 2. Load base model Llama 3.2 3B Instruct (4-bit) via Unsloth

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None  # Auto deteksi (Float16 untuk T4, Bfloat16 untuk Ampere+)
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)


## 3. Pasang adapter LoRA di atas base model

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                       "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)


## 4. Upload & load dataset JSON kamu

Ini bagian yang kamu perlu di sini: cukup **buat file JSON** berisi list `{"instruction":..., "input":..., "output":...}`, semua proses loading & formatting sudah otomatis di bawah ini.

In [ ]:
from google.colab import files
import json

print("Silakan upload file dataset kamu (contoh: dataset_contoh.json)")
uploaded = files.upload()
dataset_filename = list(uploaded.keys())[0]
print(f"File terupload: {dataset_filename}")

with open(dataset_filename, "r", encoding="utf-8") as f:
    raw_data = json.load(f)

print(f"Jumlah data: {len(raw_data)}")
print("Contoh 1 data:")
print(raw_data[0])


## 5. Format dataset ke template chat Llama 3.2 + system prompt persona Minci

Bagian ini otomatis mengubah tiap `{instruction, input, output}` jadi format percakapan lengkap dengan system prompt yang mendefinisikan kepribadian Minci.

In [ ]:
from datasets import Dataset

SYSTEM_PROMPT = (
    "Kamu adalah Minci, asisten virtual akademik dari STT Cipasung. "
    "Gaya bicaramu santai, ramah, ceria, dan luwes ala anak muda (gen-z), tapi tetap sopan dan tidak berlebihan. "
    "Kamu membantu mahasiswa dan calon mahasiswa seputar informasi PMB (Penerimaan Mahasiswa Baru) dan KRS (Kartu Rencana Studi). "
    "Jawab berdasarkan konteks dokumen yang diberikan. Jika informasi tidak ada di konteks, katakan dengan jujur bahwa kamu tidak punya info itu dan sarankan hubungi bagian Tata Usaha. "
    "Jawablah singkat, jelas, rapi, dan hangat."
)

def format_example(example):
    user_content = example["instruction"]
    if example.get("input"):
        user_content += "\n" + example["input"]

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_content},
        {"role": "assistant", "content": example["output"]},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return {"text": text}

hf_dataset = Dataset.from_list(raw_data)
hf_dataset = hf_dataset.map(format_example)

print(hf_dataset[0]["text"])


## 6. Setup trainer (SFTTrainer dari TRL)

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = hf_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    packing = False,
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 15,               # [UBAHAN 1] Dinaikkan agar AI tidak kaget di awal
        num_train_epochs = 3,
        learning_rate = 5e-5,            # [UBAHAN 2] Diturunkan drastis dari 2e-4
        max_grad_norm = 0.3,             # [UBAHAN 3] Ditambahkan untuk memotong error yang melonjak
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",    # [UBAHAN 4] Diubah dari linear ke cosine
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
    ),
)

## 7. Mulai training

Dengan dataset kecil (puluhan-ratusan contoh) di GPU T4, ini biasanya cuma butuh beberapa menit.

In [ ]:
trainer_stats = trainer.train()


## 8. Test cepat hasil fine-tuning (opsional tapi disarankan)

In [ ]:
FastLanguageModel.for_inference(model)

test_messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": "Min, gimana cara isi KRS ya?"},
]
inputs = tokenizer.apply_chat_template(test_messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
outputs = model.generate(input_ids=inputs, max_new_tokens=200, temperature=0.7, do_sample=True)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))


## 9. Merge LoRA ke base model & export langsung ke format GGUF (siap pakai Ollama)

Ini akan menghasilkan file `.gguf` terkuantisasi Q4_K_M — ukurannya kecil (~2GB) dan cocok untuk laptop dengan RAM/VRAM terbatas.

In [ ]:
model.save_pretrained_gguf(
    "minci-llama3.2-3b",
    tokenizer,
    quantization_method = "q4_k_m",
)


## 10. Download file GGUF ke laptop kamu

In [ ]:
import glob
from google.colab import files

gguf_files = glob.glob("minci-llama3.2-3b_gguf/*.gguf")
print("File GGUF ditemukan:", gguf_files)

for f in gguf_files:
    files.download(f)


---
### Selesai! 🎉
File `.gguf` yang ter-download itu yang nanti kamu pakai untuk membuat model Ollama lokal di laptop, dengan `Modelfile` yang sudah disiapkan (lihat folder `ollama/` di project).
